## 1. Setup

In [ ]:
PIPELINE_NAME <- "snt_dhis2_incidence"

### 1.0. Fallback parameters

### ⚠️⚠️⚠️ TEMP FOR DEV !!! ⚠️⚠️⚠️

In [ ]:
# USE_CARESEEKING_FROM_FILE <- FALSE # Now use this as temporary switch to simulate presence of user provided file
# # Simuate pipeline paramter for careseeking file path
# # To be deleted after pipeline.py is ready to handle this!

# if (exists("CARESEEKING_FILE_PATH")) rm(CARESEEKING_FILE_PATH)

# if (USE_CARESEEKING_FROM_FILE) {
#   # uploads/NER_careseeking_template_TEST.csv
#   CARESEEKING_FILE_PATH <- "~/workspace/uploads/NER_careseeking_template_TEST.csv"
# #   print(CARESEEKING_FILE_PATH)
# }

# if (exists("CARESEEKING_FILE_PATH")) {
#     print(paste("careseeking file path provided:", CARESEEKING_FILE_PATH))
# } else {
#     print("No careseeking file path provided.")
# }

# rm(USE_CARESEEKING_FROM_FILE) # Clean up temporary variable

In [ ]:
# ----- ⚡ Defined in pipeline.py code ---------------
if (!exists("N1_METHOD")) N1_METHOD <- "SUSP-TEST" # ⚡ For N1 calculations: use `SUSP-TEST` or `PRES`
if (!exists("ROUTINE_DATA_CHOICE")) ROUTINE_DATA_CHOICE <- "raw" # "raw" "raw_without_outliers" "imputed"
if (!exists("USE_TRANSFORMED_POPULATION")) USE_TRANSFORMED_POPULATION <- FALSE # ⚡ USE_TRANSFORMED_POPULATION bool 

if (!exists("CARESEEKING_FILE_PATH")) CARESEEKING_FILE_PATH <- NULL

# 👥 Population Disaggregation 
# Only for countries in which disaggregated data is available. Pipeline fails if you select something you don't have.
if (!exists("DISAGGREGATION_SELECTION")) DISAGGREGATION_SELECTION <- NULL # Options: "PREGNANT_WOMAN", "UNDER_5", ... 
# Disaggregation options set in pipeline.py parameters, based on 
# https://bluesquare.atlassian.net/browse/SNT25-363?focusedCommentId=85587

### 1.1. Run setup

#### `source()`

In [ ]:
source(file.path("~/workspace", "pipelines", PIPELINE_NAME, "utils", paste0(PIPELINE_NAME, ".r")))

In [ ]:
load_utils()
setup_paths()
create_intermediate_data_dir()

In [ ]:
# List required pcks
required_packages <- c("arrow", "tidyverse", "stringi", "jsonlite", "httr", "reticulate", "glue")
install_and_load(required_packages)

In [ ]:
set_env_openhexa()

### 1.2. Load and check `config` file

**Checks for SNT mandatory configuration fields**

In [ ]:
import_config_json()
config_generic()
config_incidence()
set_fixed_cols()

## 2. Load Data

### 2.1. **Routine** data (DHIS2) (parametrized choice)

In [ ]:
select_routine_dataset_and_filename()

In [ ]:
load_dhis2_routine_data()

#### Checks

... on routine data columns

In [ ]:
# `fixed_cols`: Fixed columns that should be always present regardless of the config.
check_fixed_cols_in_routine()

# `DHIS2_INDICATORS`: Indicators, as defined in the config.json file, 
#   are expected to be present if the extraction pipeline and this pipeline are run on the same config settings.
check_dhis2_indicators_cols_in_routine()

... on `N1_METHOD` selected: <br>
_**if**_ `N1_METHOD == PRES` then `PRES` must exist in config.json file _and_ in routine data <br>
_**else**_ N1 will use `SUSP-TEST` instead

In [ ]:
check_PRES_col()

### 2.2. **Population** data at level `ADM2` x `YEAR`

In [ ]:
load_population_data()

### 👥 Population Disaggregation logic

In [ ]:
# 👥 Resolve col names for disaggregated values
prepare_disaggregated_indicators(dhis2_routine, DISAGGREGATION_SELECTION, N1_METHOD)

In [ ]:
# 👥 Replace col `POPULATION` with selected disaggregated population col (if found)
# (aka "mapping")
dhis2_population_adm2 <- select_population_column(dhis2_population_adm2, DISAGGREGATED_INDICATORS_FOUND, DISAGGREGATION_SELECTION)

#### 2.2.1 **Population** data columns selection.


In [ ]:
dhis2_population_adm2 <- dhis2_population_adm2 |> 
select(YEAR, ADM1_NAME, ADM1_ID, ADM2_NAME, ADM2_ID, POPULATION) 

dim(dhis2_population_adm2)
head(dhis2_population_adm2, 3)

### 2.3. (optional) **Care Seeking Behaviour** (CSB) 
Aka "taux recherche soins"

In [ ]:
# Logic: 
# 1. check if the careseeking file exists. 
#   If found, we will load it (and use it for N3 calculations later);
#   if not found, we will issue a warning & then 
# 2. Look for DHS data.
#   If DHS data is found, we will use it for careseeking adjustment and N3 calculations.
#   If DHS data is not found, we will issue a warning and skip N3 calculations.

# Initialize
careseeking_data               <- NULL
IMPORTED_CARESEEKING_FROM_FILE <- FALSE
IMPORTED_CARESEEKING_FROM_DHS  <- FALSE

# 1. Try the specific File Path (user provided csv file)
path_to_check <- get0("CARESEEKING_FILE_PATH") # returns NULL if CARESEEKING_FILE_PATH does not exist, instead of throwing an error
# Ensure it is a valid, non-empty character string before calling file.exists
if (is.character(path_to_check) && length(path_to_check) > 0 && !is.na(path_to_check) && nzchar(path_to_check) && file.exists(path_to_check)) {
    careseeking_data <- read_csv(path_to_check, show_col_types = FALSE)
    log_msg(paste("✅ Careseeking data loaded from file:", path_to_check))
    IMPORTED_CARESEEKING_FROM_FILE <- TRUE
    careseeking_file_name          <- basename(path_to_check)
}

# 2. Try DHS Fallback (if file loading failed). DHS data could come from SNT pipeline (if available to that country)
if (is.null(careseeking_data)) {
    log_msg("Careseeking data file not found or path missing. Looking at DHS data instead...", "warning")
    careseeking_data <- load_dhs_careseeking_data()
    
    if (!is.null(careseeking_data)) {
        IMPORTED_CARESEEKING_FROM_DHS <- TRUE
    } else {
        log_msg("All careseeking data sources failed. 🦘 N3 calculations will be skipped.", "warning")
    }
}

head(careseeking_data, 5)

#### Validate careseeking data

In [ ]:
# Check and validate careseeking data, and standardize column names for downstream processing
careseeking_data <- validate_and_format_careseeking_data(
    careseeking_data, 
    IMPORTED_CARESEEKING_FROM_FILE, 
    IMPORTED_CARESEEKING_FROM_DHS
)

head(careseeking_data, 5)

### 2.4. Load Reporting Rate 

Import Reporting Rate file based on what is available in the latest OH Dataset version (which depends on last run reporting rate pipepline).

📅 **Important**: reporting rate must be **monthly**!

In [ ]:
load_reporting_rate_data()

#### 🔍 Checkon data completeness for `REPORTING_RATE` data
Normally we should have "complete" data (no missing or `NA` values). However, when using certain datasets (from pipeline: "Reporting Rate (Dataset)") we might have incomplete coverage and hence `NA`s ... <br>
These are "problematic" because **N2** (Incidence adj 2) will also have `NA` values.

In [ ]:
check_reporting_rate_data()

-------------------------------

## 3. Calculate Incidence
First calculate monthly cases, then yearly incidence.

### 3.1 **Monthly cases**


These methods follow the standard WHO approach for estimating malaria incidence from routine health information systems (WHO, 2023).
As shown in the code, we begin by calculating **monthly malaria case metrics** (confirmed, tested, presumed) at the **ADM2** level and join them with the **monthly reporting rate**. 

This allows us to compute the **test positivity rate** (TPR, where `TPR` = `CONF` / `TEST`) and adjust for incomplete testing using the formula: 
> **N1** = `CONF` + (`PRES` × `CONF` / `TEST`)

Which is equivalent to:
> **N1** = `CONF` + (`PRES` × **TPR**)

where:
- **N1** = cases adjusted for testing gaps 
- `CONF` = **confirmed** cases
- `PRES` = **presumed** cases (either `SUSP` - `TEST` or directly available as `PRES`) 👈 this is a parameter (`N1_METHOD`)
- `TEST` = **tested** cases 
- **TPR** = Test Positivity Rate (`CONF` / `TEST`)
  
This produces `N1`, the number of cases adjusted for testing gaps, calculated at the monthly level in line with WHO recommendations to capture intra-annual variation.

Next, we adjust for incomplete reporting using: 
> **N2** = **N1** / `REPORTING_RATE`

where `REPORTING_RATE` is at the monthly levele, and is the ratio of received reports (submission to DHIS2) divided by the expected reports.

Finally, _if_ **careseeking** data is **available**, N3 is calculated as follows:
> **N3** = N2 / `CARESEEKING`

where `CARESEEKING` = user-provided metric of careseeking behaviour (as proportion)

NB: this is equivalent to the older: 
> **N3** = N2 + (N2 * proportion of U5 treated in the **private** sector / proportion of U5 treated in the **public** sector) + (N2 * proportion of U5 which **did not receive any treatment** / proportion of U5 treated in the **public** sector)

and assumes the same TPR across all sectors (private and public).



**Important note**<br>
In case reporting rate equals zero (none of the health facilities reported in a given month), N2 is set to `NA`. Note that the annual N2 will be underestimated, which is preferable compared to having `Inf` values.

In [ ]:
enforce_numeric_cols()

#### 3.1.0. Aggregate at `ADM2` x `MONTH` & calculate **TPR**

In [ ]:
# Group & compute TPR
monthly_cases <- routine_data |>
    group_by(ADM1_ID, ADM2_ID, YEAR, MONTH) |> # ADM1 is needed to join careseeking data
    summarise(
      CONF = sum(CONF, na.rm = TRUE),
      TEST = sum(TEST, na.rm = TRUE),
      SUSP = sum(SUSP, na.rm = TRUE),
      across(any_of("PRES"), ~sum(., na.rm = TRUE), .names = "PRES"), # to handle missing 'PRES' column gracefully
      .groups = "drop") |>
    # Cleaning TEST data for "SUSP-TEST" method
    mutate(TEST = ifelse(N1_METHOD == "SUSP-TEST" & !is.na(SUSP) & (TEST > SUSP), SUSP, TEST)) |>
    left_join(reporting_rate_data,
              by = c("ADM2_ID", "YEAR", "MONTH")) |>   
    # Calculate TPR based on CONF and TEST
    # Note: if TEST is 0 or NA, set TPR = 1 (to avoid division by zero which produces Inf)
    mutate( 
      TPR = ifelse(!is.na(CONF) & !is.na(TEST) & (TEST != 0), CONF / TEST, 1)
    )

#### 3.1.1. Calculate **N1**

In [ ]:
# Calculate N1 based on `N1_METHOD` & availability of `PRES` 

if (N1_METHOD == "SUSP-TEST") {
    monthly_cases <- monthly_cases %>%
      mutate(N1 = CONF + ((SUSP - TEST) * TPR))
      log_msg("Calculating N1 as `N1 = CONF + ((SUSP - TEST) * TPR)`")
} else if (N1_METHOD == "PRES") {
    # if: column named "PRES" exists in `monthly_cases` and contains at least one non-missing value
    if ("PRES" %in% names(monthly_cases) && !all(is.na(monthly_cases$PRES))) {
      monthly_cases <- monthly_cases %>%
        mutate(N1 = CONF + (PRES * TPR))
        log_msg("ℹ️ Calculating N1 as `N1 = CONF + (PRES * TPR)`")
    } else {
      log_msg("🚨 Warning: 'PRES' not found in routine data or contains all `NA` values! 🚨 Calculating N1 using 'SUSP-TEST' method instead.")
      monthly_cases <- monthly_cases %>%
        mutate(N1 = CONF + ((SUSP - TEST) * TPR))
    }
} else {
    log_msg("Invalid N1_METHOD. Please use 'PRES' or 'SUSP-TEST'.") # not really necessary ... 
}

#### 3.1.2. Calculate **N2**

In [ ]:
# Calculate N2
monthly_cases <- monthly_cases %>%
    mutate(
      N2 = ifelse(REPORTING_RATE == 0, NA_real_, N1 / REPORTING_RATE) # On the fly convert `RR == 0` to NA to avoid N2 == Inf
    )

In [ ]:
handle_zeros_in_reporting_rate()

#### 3.1.4. (optional) Calculate **N3**

In [ ]:
# Only calculate N3 if CARESEEKING data is avaiable 
if (!is.null(careseeking_data)) {
    # join careseeking data to monthly_cases (at ADM1 level, if careseeking data is at ADM1; at ADM2 level if careseeking data is at ADM2)
    monthly_cases <- join_careseeking_data(monthly_cases, careseeking_data)

    # Now calculate N3 using the joined careseeking data
    monthly_cases <- monthly_cases |>
    mutate(
        # N3 = N2 + (N2 * PCT_PRIVATE_CARE / PCT_PUBLIC_CARE) + (N2 * PCT_NO_CARE / PCT_PUBLIC_CARE) # OLD formula (unnecessarily complicated)
        # N3 = N2 / PCT_PUBLIC_CARE # Simplified
        # N3 = N2 / CARESEEKING_PCT # Equivalent, generic
        N3 = ifelse(CARESEEKING_PCT == 0, NA_real_, N2 / (CARESEEKING_PCT / 100)) # avoid N2 == Inf
        # PS: yes, it is a bit silly to first expect careseeking data to be in percentage format (0-100) 
        # and then convert it back to a proportion (0-1) for the calculation. 
        # But this is based on expecting DHS data, which is in the percentage (0-100, "PCT") format.
    )
    log_msg("✅ N3 calculated using careseeking data.")
} else {
    print("🦘 Careseeking data not available, skipping calculation of N3.")
}

In [ ]:
head(monthly_cases, 3)

#### 💾 Export `monthly_cases` (for 📓report notebook)
For coherence checks, which need monthly resolution ... !

In [ ]:
export_monthly_cases(monthly_cases)

### 🔍 Data **coherence** checks on **monthly cases**
Check for ratios or differences that will cause negative values -> which will causes adjusted incidence to be lower than the values it adjust


Namely, the following relationships among INDICATORs:
* SUSP-TEST
* CONF/TEST
* N1 == CONF ... (when PRES == 0)

#### 1. `PRES == 0`: causes `N1 == CONF` 
(if `N1_METHOD == "PRES"`)

In [ ]:
coherence_check_PRES(monthly_cases)

#### 2. `SUSP-TEST`: if negative, then N1 smaller or equal to CONF (ADJ =< CRUDE)
(if `N1_METHOD == "SUSP-TEST"`)

In [ ]:
coherence_check_SUSP_TEST(monthly_cases)

#### 3. `CONF/TEST` = `TPR` (to calculate N1: Incidence adjusted for **Testing**)
This **ratio should** always be **≤ 1** because **there should _not_ be more confirmed cases than tested** ...

(but if very small, then N1 could be smaller or equal to CONF (so ADJ INC ≤ CRUDE))

In [ ]:
coherence_check_CONF_TEST(monthly_cases)

### 3.2 **Yearly incidence**
After calculating N1 and N2 for each `ADM2`-`MONTH`, we aggregate the data annually to compute the yearly totals (sums) for crude cases (`CONF`), `N1` and `N2`. Finally, we compute:
* Crude incidence: C / POP × 1000
* Incidence adjusted for testing: N1 / POP × 1000
* Incidence adjusted for testing and reporting: N2 / POP × 1000
* Incidence adjusted for testing, reporting and careseeking behaviour (optional): N3 / POP × 1000

In [ ]:
# ---- 1. Enforce column types upfront ----
monthly_cases <- monthly_cases %>% 
    mutate(across(where(is.numeric), as.numeric))  # Convert all numeric columns
  
population_data <- dhis2_population_adm2 %>% # population_data
    mutate(across(c(YEAR, POPULATION), as.numeric))

In [ ]:
# ---- 2. Core calculation ----
yearly_incidence <- monthly_cases %>%
    group_by(ADM2_ID, YEAR) %>%
    summarise(
        # 🚨 removed `na.rm = TRUE` on 20250702 - if things break check here! 🚨 
      across(c(CONF, N1, N2), ~sum(.)), #, na.rm = TRUE)), # 🔍 PROBLEM: if NA's in N2 (due to missing RR data), the sum of N2 by YEAR is smaller than the sum of N1 !
      .groups = "drop"
    ) %>%
    left_join(
      population_data,
      by = c("ADM2_ID", "YEAR")
    ) %>%
    mutate(
      INCIDENCE_CRUDE = CONF / POPULATION * 1000,
      INCIDENCE_ADJ_TESTING = N1 / POPULATION * 1000,
      INCIDENCE_ADJ_REPORTING = N2 / POPULATION * 1000
    ) |>
    ungroup()

In [ ]:
# careseeking_data
monthly_cases |> head()

In [ ]:
# ---- Add empty INCIDENCE_ADJ_CARESEEKING col in case careseeking data is not available ---- 
if (!is.null(careseeking_data) && "N3" %in% names(monthly_cases)) {
    n3_data <- monthly_cases %>%
      group_by(ADM2_ID, YEAR) %>%
      summarise(N3 = sum(N3, na.rm = TRUE),
                .groups = "drop") |>
      ungroup()
    
    yearly_incidence <- yearly_incidence %>%
      left_join(n3_data, by = c("ADM2_ID", "YEAR")) %>%
      mutate(
        INCIDENCE_ADJ_CARESEEKING = N3 / POPULATION * 1000
      )
  } else {
    yearly_incidence <- yearly_incidence |>
      mutate(
        INCIDENCE_ADJ_CARESEEKING = NA
            )
  }

In [ ]:
head(yearly_incidence, 3)

### 🔍 Data **coherence** checks on **yearly incidence**
Here we check if values of Indicidence (already at `YEAR` resolution) make sense in relation to each other.<br>
Namely:
* crude values should be the lowest, and any consecutive **adjustment** should cause the incidence values to **increase** or remain the **same** - but should never be lower!

#### 1. `INCIDENCE_ADJ_TESTING` (adj. level 1) should always be greater than `INCIDENCE_CRUDE` (not adjusted)

In [ ]:
coherence_checkes_yearly_incidence(yearly_incidence, incidence_col_1 = "INCIDENCE_CRUDE", incidence_col_2 = "INCIDENCE_ADJ_TESTING")

#### 2. `INCIDENCE_ADJ_REPORTING` (adj. level 2) should always be greater than `INCIDENCE_ADJ_TESTING` (adj. level 1)

In [ ]:
coherence_checkes_yearly_incidence(yearly_incidence, incidence_col_1 = "INCIDENCE_ADJ_TESTING", incidence_col_2 = "INCIDENCE_ADJ_REPORTING")

## 4. Export to `/data/dhis2_incidence/` folder

### 4.0. Keep only essential cols 
Based on [SNT Pipelines Data glossary](https://docs.google.com/spreadsheets/d/1qvZMsmCWU6cVLgGZTEXsd5xmoecIxb4LAd-g_2qzYdw/edit?usp=sharing)

In [ ]:
yearly_incidence <- yearly_incidence |>
select(
    YEAR, 
    starts_with("ADM"),
    starts_with("POPULATION"),
    starts_with("INCIDENCE")
)

In [ ]:
yearly_incidence |> head()

#### 👥 Population Disaggregation logic 

Provide a msg to the user to indicate that the results correspond to a specific version of indicators and population (under5, pregnant or totals).

In [ ]:
if (DISAGGREGATED_INDICATORS_FOUND) {
    log_msg(glue("ℹ️ The results have been computed using the following Indicators: {paste(target_colnames, collapse=', ')}"))
    log_msg(glue("ℹ️ The results have been computed using the following Population: {POPULATION_SELECTION}"))
}

In [ ]:
# Tell the user what data and col were used for careseeking adjustment, if applicable
if (!is.null(careseeking_data) && "N3" %in% names(monthly_cases)) {
    second_part_of_log <- if (IMPORTED_CARESEEKING_FROM_FILE) {
        glue("user-provided file.")
    } else if (IMPORTED_CARESEEKING_FROM_DHS) {
        glue("DHS data {careseeking_file_name}.")
    } 
    log_msg(glue("INCIDENCE_ADJ_CARESEEKING has been computed after adjusting INCIDENCE_ADJ_REPORTING for careseeking data from the {second_part_of_log}"))
} else {
    log_msg("No careseeking adjustment applied since careseeking data was not available. INCIDENCE_ADJ_CARESEEKING column has been created with NA values.")
}

In [ ]:
# Export the data

# CSV
save_yearly_incidence(yearly_incidence, DATA_PATH, ".csv", write_csv)

# Parquet
save_yearly_incidence(yearly_incidence, DATA_PATH, ".parquet", arrow::write_parquet)